In [2]:
import matplotlib.pyplot as plt
%load_ext autoreload
%autoreload 2

In [3]:
import re
from collections import Counter, defaultdict
import plotly.graph_objects as go
import json
from pathlib import Path

In [4]:
def _iter_metadata_records(payload):
    if isinstance(payload, dict):
        yield payload
        for key in ('hits', 'records', 'datasets', 'results', 'entries', 'items'):
            seq = payload.get(key)
            if isinstance(seq, list):
                for item in seq:
                    yield from _iter_metadata_records(item)
    elif isinstance(payload, list):
        for item in payload:
            yield from _iter_metadata_records(item)

def _extract_recid_from_record(record):
    candidates = [
        record.get('recid'),
        record.get('id'),
        (record.get('metadata') or {}).get('recid'),
        (record.get('metadata') or {}).get('id'),
    ]
    for cand in candidates:
        if cand is None:
            continue
        try:
            return int(cand)
        except (TypeError, ValueError):
            continue
    return None

records_from_lists = {}
lists_dir = Path('../lists')
if lists_dir.exists():
    for path in sorted(lists_dir.glob('*')):
        if not path.is_file():
            continue
        raw_text = path.read_text(encoding='utf-8').strip()
        if not raw_text:
            continue
        decoded = None
        try:
            decoded = json.loads(raw_text)
        except json.JSONDecodeError:
            pass
        if decoded is None:
            for line in raw_text.splitlines():
                line = line.strip()
                if not line:
                    continue
                try:
                    decoded = json.loads(line)
                except json.JSONDecodeError:
                    decoded = None
                if decoded is not None:
                    for rec in _iter_metadata_records(decoded):
                        recid = _extract_recid_from_record(rec)
                        if recid is not None and recid not in records_from_lists:
                            records_from_lists[recid] = rec
                    decoded = None
            continue
        for rec in _iter_metadata_records(decoded):
            recid = _extract_recid_from_record(rec)
            if recid is not None and recid not in records_from_lists:
                records_from_lists[recid] = rec

In [5]:
# print values of date_created, formats, number_events, number_files, size, type - secondary, collision_information - energy, categories - primary
for recid in list(records_from_lists.keys())[420:422]:
    metadata = records_from_lists[recid]
    # metadata = record.get('metadata', {})
    date_created = metadata.get('date_created')
    distributions = metadata.get('distribution', {})
    formats = distributions.get('formats')
    number_events = distributions.get('number_events')
    number_files = distributions.get('number_files')
    size = distributions.get('size')
    record_type = metadata.get('type')
    collision_info = metadata.get('collision_information', {})
    energy = collision_info.get('energy')
    categories = metadata.get('categories', {})
    primary_category = categories.get('primary')
    print(f"Recid: {recid}")
    print(f"  Date Created: {date_created}")    #['1994']
    print(f"  Formats: {formats}")  #['SHORT']
    print(f"  Number of Events: {number_events}")   #10500
    print(f"  Number of Files: {number_files}") #7
    print(f"  Size: {size}") #223534080
    print(f"  Type: {record_type}") #{'primary': 'Dataset', 'secondary': ['Simulated']}
    print(f"  Collision Energy: {energy}") #89-94 GeV
    print(f"  Primary Category: {primary_category}") #2 Fermion

Recid: 81302
  Date Created: ['1994']
  Formats: ['SHORT']
  Number of Events: 10500
  Number of Files: 7
  Size: 223534080
  Type: {'primary': 'Dataset', 'secondary': ['Simulated']}
  Collision Energy: 89-94 GeV
  Primary Category: 2 Fermion
Recid: 81220
  Date Created: ['1994']
  Formats: ['SHORT']
  Number of Events: 1998
  Number of Files: 10
  Size: 57553920
  Type: {'primary': 'Dataset', 'secondary': ['Simulated']}
  Collision Energy: 89-94 GeV
  Primary Category: Higgs


In [11]:
# Sankey chart visualizing dataset counts across key metadata dimensions
import re
from collections import Counter, defaultdict
import plotly.graph_objects as go

def _normalize_file_entry(raw):
    checksum = raw.get('checksum') or raw.get('checksum_value')
    checksum_type = raw.get('checksum_type')
    if isinstance(checksum, str) and ':' in checksum and not checksum_type:
        checksum_type, checksum = checksum.split(':', 1)
    remote = raw.get('uri') or raw.get('remote') or raw.get('url') or raw.get('link')
    size = raw.get('size') or raw.get('bytes') or raw.get('filesize')
    return {
        'label': raw.get('name') or raw.get('filename') or raw.get('path'),
        'remote': remote,
        'local_path': raw.get('local_path') or raw.get('local'),
        'size': size,
        'checksum_type': checksum_type,
        'checksum': checksum,
        'downloaded': bool(raw.get('downloaded', False)),
    }

def _extract_files(record):
    meta = record.get('metadata') or {}
    files = meta.get('files') or record.get('files') or []
    normalized = []
    for item in files:
        if isinstance(item, dict):
            normalized.append(_normalize_file_entry(item))
    return normalized

def _coerce_year(meta):
    for key in ("date_created", "date", "record_creation_date"):
        raw = meta.get(key)
        if isinstance(raw, list):
            raw = raw[0] if raw else None
        if isinstance(raw, str):
            match = re.search(r"\d{4}", raw)
            if match:
                return match.group(0)
    return "Unknown"

def _bucket_events(value):
    try:
        amount = float(value)
    except (TypeError, ValueError):
        amount = None
    if amount is None:
        return "Unknown events"
    if amount == 0:
        return "0"
    if amount <= 1e3:
        return "≤1k"
    if amount <= 1e5:
        return "1k–100k"
    if amount <= 1e7:
        return "100k–10M"
    return ">10M "

def _bucket_files(count):
    if count is None:
        return "Unknown files"
    if count == 0:
        return "0"
    if count <= 20:
        return "1–20"
    if count <= 100:
        return "21–100"
    if count <= 1000:
        return "101–1000"
    return ">1000"

def _bucket_size(size_bytes):
    try:
        size_gb = float(size_bytes) / (1024 ** 3)
    except (TypeError, ValueError):
        return "Unknown size"
    if size_gb < 1:
        return "<1 GB"
    if size_gb < 10:
        return "1–10 GB"
    if size_gb < 100:
        return "10–100 GB"
    return "≥100 GB"

def _safe_first(value, fallback="Unknown"):
    if value is None:
        return fallback
    if isinstance(value, list):
        return value[0] if value else fallback
    return value

def _resolve_metadata(record):
    return record.get("metadata") if isinstance(record, dict) and record.get("metadata") else record

def _year_sort_key(label):
    text = str(label) if label is not None else "Unknown"
    m = re.search(r"\b(\d{4})\b", text)
    if m:
        return (0, int(m.group(1)), text.lower())
    return (1, float("inf"), text.lower())

def _energy_sort_key(label):
    text = str(label) if label is not None else "Unknown"
    low = text.lower()
    if low.startswith("unknown"):
        return (1, float("inf"), float("inf"), low)
    nums = re.findall(r"\d+(?:\.\d+)?", text)
    if not nums:
        return (1, float("inf"), float("inf"), low)
    lo = float(nums[0])
    hi = float(nums[1]) if len(nums) > 1 else lo
    return (0, lo, hi, low)

def _default_sort_key(label):
    text = str(label) if label is not None else "Unknown"
    low = text.lower()
    return (low.startswith("unknown"), low)

tier_sequence = [
    ("date", "Year"),
    ("category", "Category"),
    ("secondary", "Type"),
    ("energy", "Collision Energy"),
    ("format", "Format"),
    ("events", "Number of Events"),
    ("files", "Number of Files"),
    ("size", "Size"),
]
tier_positions = {
    key: (idx / (len(tier_sequence) - 1) if len(tier_sequence) > 1 else 0.5)
    for idx, (key, _) in enumerate(tier_sequence)
}

NODE_PALETTE = [
    "#4C78A8", "#F58518", "#54A24B", "#E45756", "#72B7B2",
    "#B279A2", "#FF9DA6", "#9D755D", "#BAB0AC", "#2E91E5",
    "#E15F99", "#1CA71C", "#FB0D0D", "#DA16FF", "#222A2A",
]

def _hex_to_rgba(hex_color, alpha=0.45):
    c = hex_color.lstrip("#")
    if len(c) != 6:
        return f"rgba(120,120,120,{alpha})"
    r, g, b = int(c[0:2], 16), int(c[2:4], 16), int(c[4:6], 16)
    return f"rgba({r},{g},{b},{alpha})"

node_labels, node_index = [], {}
node_x = []
node_colors = []
tier_values = defaultdict(list)
link_counter = Counter()
summary = {"categories": Counter(), "formats": Counter(), "dates": Counter()}

for recid, raw_record in records_from_lists.items():
    metadata = _resolve_metadata(raw_record) or {}
    dist = metadata.get("distribution") or {}
    type_info = metadata.get("type") or {}
    collision_info = metadata.get("collision_information") or {}
    categories = metadata.get("categories") or {}

    files = _extract_files({"metadata": metadata})
    file_count = len(files) if files else dist.get("number_files")
    total_size = (
        sum(float(f.get("size")) for f in files if f.get("size") not in (None, ""))
        if files else dist.get("size")
    )

    formats = dist.get("formats") or metadata.get("formats")
    if isinstance(formats, list) and formats:
        format_label = formats[0] if len(formats) == 1 else f"{formats[0]} (+{len(formats)-1})"
    else:
        format_label = _safe_first(formats)

    features = {
        "date": _coerce_year(metadata),
        "category": _safe_first(categories.get("primary")),
        "secondary": _safe_first(type_info.get("secondary") if isinstance(type_info, dict) else type_info),
        "energy": _safe_first(collision_info.get("energy")),
        "format": format_label,
        "events": _bucket_events(dist.get("number_events") or metadata.get("number_events")),
        "files": _bucket_files(file_count),
        "size": _bucket_size(total_size),
    }

    for key in ("category", "formats", "date"):
        pass  # placeholder; summary fills below

    summary["categories"][features["category"]] += 1
    summary["formats"][features["format"]] += 1
    summary["dates"][features["date"]] += 1

    path = []
    for key, _ in tier_sequence:
        value = features[key] or "Unknown"
        path.append((key, value))
        if value not in tier_values[key]:
            tier_values[key].append(value)

    for (src_key, src_val), (dst_key, dst_val) in zip(path, path[1:]):
        link_counter[((src_key, src_val), (dst_key, dst_val))] += 1

tier_sorters = {
    "date": _year_sort_key,
    "energy": _energy_sort_key,
}
for key, _ in tier_sequence:
    tier_values[key] = sorted(tier_values[key], key=tier_sorters.get(key, _default_sort_key))

for key, display in tier_sequence:
    for value in tier_values[key]:
        node_index[(key, value)] = len(node_labels)
        node_labels.append(f"{value}")
        node_x.append(tier_positions[key])
        node_colors.append(NODE_PALETTE[len(node_colors) % len(NODE_PALETTE)])

sources, targets, values = [], [], []
link_colors = []
for (src, dst), count in link_counter.items():
    sources.append(node_index[src])
    targets.append(node_index[dst])
    values.append(count)
    link_colors.append(_hex_to_rgba(node_colors[node_index[src]], alpha=0.45))

column_annotations = [
    dict(
        x=tier_positions[key],
        y=1.,
        xref="paper",
        yref="paper",
        text=display,
        showarrow=False,
        font=dict(size=12, color="#1f2937"),
        xanchor="center",
        yanchor="bottom",
    )
    for key, display in tier_sequence
]

fig = go.Figure(data=[go.Sankey(
    node=dict(label=node_labels, pad=15, thickness=20, x=node_x, color=node_colors),
    link=dict(source=sources, target=targets, value=values, color=link_colors),
    arrangement="snap",
)])
fig.update_layout(
    template="plotly_white",
    title_text="Delphi Metadata Sankey Visualization",
    font=dict(size=12, color="#111827"),
    height=650,
    margin=dict(t=110, l=20, r=20, b=20),
    annotations=column_annotations,
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
)
fig.write_image("sankey_chart.png", width=1500, height=500, scale=2)
fig.write_html("sankey_chart.html")
fig.show()

total = len(records_from_lists)
print(f"Datasets visualized: {total}")
print("Top primary categories:", summary["categories"].most_common(5))
print("Top formats:", summary["formats"].most_common(5))
print("Top years:", summary["dates"].most_common(5))

Datasets visualized: 12745
Top primary categories: [('Higgs', 10679), ('2 Fermion', 1234), ('4 Fermion', 379), ('Unknown', 300), ('Susy', 78)]
Top formats: [('XSHORT', 11689), ('SHORT', 450), ('LONG', 284), ('DSTO', 279), ('RAWD', 25)]
Top years: [('1999', 6234), ('2000', 4307), ('1998', 1174), ('1997', 275), ('1994', 272)]
